# Approach 3: Audio Architecture Mimicry (Waveform-to-Spectrogram)

This notebook tricks an audio model into "hearing" the brain waves. We convert the 1D electrical signals into 2D pseudo-spectrograms (using Short-Time Fourier Transform) and feed them into a Deep Convolutional model mimicking the front-end of Whisper or Wav2Vec 2.0.

In [ ]:
# Requirements: pip install torch torchaudio h5py matplotlib
import glob
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio.transforms as T
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset

DATASET_PATH = r"C:\SDE Projects\Open-BCI-EEG-Waves-To-Text-Translation-And-further-Robotic-Implementaions\dataset\extracted"
MODELS_DIR = "models"
METRICS_DIR = "metrics"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

## 1. Feature Extraction (Spectrograms)
Converting time-series waves into frequency-over-time images.

In [ ]:
class EEGAudioDataset(Dataset):
    def __init__(self, data_dir, max_len=1000):
        self.files = glob.glob(os.path.join(data_dir, "**", "*.h5"), recursive=True)
        self.max_len = max_len
        self.chars = (
            "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?'-"
        )
        self.char_to_idx = {ch: i + 1 for i, ch in enumerate(self.chars)}
        self.char_to_idx["<PAD>"] = 0
        self.vocab_size = len(self.char_to_idx)

        # STFT parameters tuned for EEG (low frequencies)
        self.spectrogram = T.Spectrogram(
            n_fft=128,
            win_length=64,
            hop_length=16,
            center=True,
            pad_mode="reflect",
            power=2.0,
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        transcript = os.path.basename(file_path).replace(".h5", "").replace("_", " ")

        with h5py.File(file_path, "r") as f:
            keys = list(f.keys())
            if len(keys) > 0:
                eeg_data = f[keys[0]][:]
            else:
                eeg_data = np.zeros((105, self.max_len))

        if eeg_data.shape[1] < self.max_len:
            eeg_data = np.pad(eeg_data, ((0, 0), (0, self.max_len - eeg_data.shape[1])))
        else:
            eeg_data = eeg_data[:, : self.max_len]

        eeg_tensor = torch.tensor(eeg_data, dtype=torch.float32)  # [105, max_len]
        spec = self.spectrogram(eeg_tensor)  # [105, freq_bins, time_frames]

        # Flatten the channel and freq dimensions to create a massive "audio" feature per time frame
        # e.g., 105 channels * 65 freq bins = 6825 features per frame
        spec = spec.view(-1, spec.size(2)).transpose(0, 1)  # [time_frames, 6825]

        # Normalize
        spec = torch.log1p(spec)

        target = [self.char_to_idx.get(c, 0) for c in transcript]
        return spec, torch.tensor(target, dtype=torch.long)


dataset = EEGAudioDataset(DATASET_PATH)
print(f"Found {len(dataset)} HDF5 files for training.")

## 2. Audio-Mimicry Architecture
A CNN backbone feeding into a sequential model, analogous to Wav2Vec.

In [ ]:
class AudioMimicEEG(nn.Module):
    def __init__(self, feature_dim=6825, hidden_size=512, num_classes=65):
        super().__init__()
        # Downsampling CNN
        self.conv = nn.Sequential(
            nn.Conv1d(feature_dim, hidden_size, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Conv1d(hidden_size, hidden_size, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
        )

        # GRU for temporal mapping
        self.rnn = nn.GRU(
            hidden_size, hidden_size, num_layers=3, batch_first=True, bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x shape: [batch, time, feature_dim]
        x = x.transpose(1, 2)  # [batch, feature_dim, time]
        x = self.conv(x)
        x = x.transpose(1, 2)  # [batch, time, hidden_size]

        self.rnn.flatten_parameters()
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out

## 3. Training Loop

In [ ]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

if len(dataset) > 0:
    # Feature dim = 105 channels * (n_fft/2 + 1) = 105 * 65 = 6825
    model = AudioMimicEEG(feature_dim=6825, num_classes=dataset.vocab_size)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    epochs = 3
    loss_history = []
    model.train()
    print("Training Audio Mimicry Model...")
    for epoch in range(epochs):
        epoch_loss = 0
        for feat, target in dataloader:
            batch_size = feat.size(0)
            # Stride of 2 applied twice in CNN = divide time by 4
            input_lengths = torch.full(
                size=(batch_size,), fill_value=feat.size(1) // 4, dtype=torch.long
            )
            target_lengths = torch.tensor([len(target[0])])

            optimizer.zero_grad()
            out = model(feat)
            out = out.transpose(0, 1)
            out = nn.functional.log_softmax(out, dim=2)

            loss = criterion(out, target, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / max(1, len(dataloader))
        loss_history.append(avg_loss)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

    torch.save(
        model.state_dict(), os.path.join(MODELS_DIR, "audio_mimic_checkpoint.pth")
    )

    plt.plot(loss_history, marker="^", color="red")
    plt.title("Audio Mimicry Loss")
    plt.xlabel("Epoch")
    plt.ylabel("CTC Loss")
    plt.savefig(os.path.join(METRICS_DIR, "audio_loss_curve.png"))
    print("Training Complete!")
else:
    print("No HDF5 data found.")